# Notebook 04 — Club vs National Feature Importance (#85)

**Objective:** Use PCA loadings to rank which metrics drive the club vs national performance gap, and explain **Club-strong** vs **National-strong** specialists.

**Prerequisites:** Notebook `03_consistency_score` (scores + export) or equivalent tables in BigQuery.

**Inputs (Databricks → BigQuery foreign catalog):**
- `int_player_club_vs_national` — context z-scores
- `analytics.pca_loadings` — PCA feature weights
- `analytics.consistency_scores` (optional) — precomputed quadrants from #84

**Outputs:** Ranked feature-importance tables and plots (display only); interpretation in markdown.

**Research question:** Which tactical metrics best explain who elevates their game for club vs country? (RQ3)

---
## 0. Setup

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="muted")

BQ_CATALOG = os.environ.get("BQ_CATALOG", "bq_raw_statsbomb_sa_catalog")
BQ_PCA_LOADINGS = f"{BQ_CATALOG}.analytics.pca_loadings"
INT_TABLE = f"{BQ_CATALOG}.raw_statsbomb_intermediate.int_player_club_vs_national"
CONSISTENCY_TABLE = f"{BQ_CATALOG}.analytics.consistency_scores"

USE_PRECOMPUTED_SCORES = True  # False = recompute quadrants in this notebook

print(f"Players:     {INT_TABLE}")
print(f"Loadings:    {BQ_PCA_LOADINGS}")
print(f"Scores:      {CONSISTENCY_TABLE} (optional)")
print("Setup complete.")

---
## 1. Features and methodology

### 1.1 Features (11, same as clustering / notebook 03)

Context z-scores come from dbt (`int_player_club_vs_national`): each value is relative to **peers in that context** (club vs club, national vs national).

### 1.2 PCA weights

Same as notebook 03:

$$
w_f = \frac{\sum_i |\text{loading}_{i,f}|}{\sum_{f'} \sum_i |\text{loading}_{i,f'}|}
$$

### 1.3 Per-player feature gap (directional)

For each player and feature:

$$
\Delta z_f = z_{f,\text{national}} - z_{f,\text{club}}
$$

- **Positive** \(\Delta z_f\): stronger in **national** team context (vs club peers / national peers respectively).
- **Negative** \(\Delta z_f\): stronger in **club** context.

### 1.4 Weighted gap contribution (importance proxy)

For each player:

$$
c_f = w_f \cdot |\Delta z_f|
$$

Summing \(c_f\) approximates how much feature \(f\) contributes to that player's overall club–national shift, scaled by PCA importance.

**Cohort-level importance** (all dual-context players):

$$
I_f = \text{mean}_{\text{players}}(c_f)
$$

**Specialist comparison:** mean \(\Delta z_f\) for **Club-strong** vs **National-strong** quadrants (from notebook 03), to see which metrics separate the archetypes.

In [ ]:
FEATURES = [
    "shots_per_90",
    "xg_per_90",
    "xg_per_shot",
    "dribbles_per_90",
    "carries_att_third_per_90",
    "passes_att_third_per_90",
    "pass_completion_pct",
    "pressures_per_90",
    "interceptions_per_90",
    "clearances_per_90",
    "aerial_duels_per_90",
]
Z_COLS = [f"z_{f}" for f in FEATURES]
FEATURE_LABELS = {
    "shots_per_90": "Shots p90",
    "xg_per_90": "xG p90",
    "xg_per_shot": "xG / shot",
    "dribbles_per_90": "Dribbles p90",
    "carries_att_third_per_90": "Att. third carries p90",
    "passes_att_third_per_90": "Att. third passes p90",
    "pass_completion_pct": "Pass completion %",
    "pressures_per_90": "Pressures p90",
    "interceptions_per_90": "Interceptions p90",
    "clearances_per_90": "Clearances p90",
    "aerial_duels_per_90": "Aerial duels p90",
}

---
## 2. Load data

In [ ]:
players = spark.sql(f"SELECT * FROM {INT_TABLE}").toPandas()
loadings = spark.sql(
    f"SELECT component, feature, loading FROM {BQ_PCA_LOADINGS}"
).toPandas()


def normalize_is_international(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    if isinstance(val, (bool, np.bool_)):
        return bool(val)
    if isinstance(val, (int, float)):
        return bool(val)
    s = str(val).strip().lower()
    if s in ("true", "1", "t", "yes", "national"):
        return True
    if s in ("false", "0", "f", "no", "club"):
        return False
    raise ValueError(f"Unrecognized is_international: {val!r}")


players["is_international"] = players["is_international"].apply(normalize_is_international)
players = players.dropna(subset=["is_international"]).copy()
players["is_international"] = players["is_international"].astype(bool)

agg_cols = {"player_name": "first", **{c: "first" for c in Z_COLS}}
players = (
    players.groupby(["player_id", "is_international"], as_index=False)
    .agg(agg_cols)
)

# Spark foreign catalog can return duplicate column names (22 cols instead of 11 z_*)
players = players.loc[:, ~players.columns.duplicated()].copy()
missing_z = [c for c in Z_COLS if c not in players.columns]
if missing_z:
    raise ValueError(f"Missing z columns in {INT_TABLE}: {missing_z}")

weights = (
    loadings.groupby("feature")["loading"]
    .apply(lambda s: s.abs().sum())
    .reindex(FEATURES)
)
weights = weights / weights.sum()
w = weights.to_dict()

print(f"Players (2 rows each): {len(players):,} | distinct: {players['player_id'].nunique():,}")
display(loadings.head())

---
## 3. Build per-player z gaps and weighted contributions

In [ ]:
Z_RENAME = dict(zip(Z_COLS, FEATURES))


def z_by_context(df: pd.DataFrame, is_national: bool) -> pd.DataFrame:
    mask = df["is_international"] if is_national else ~df["is_international"]
    out = (
        df.loc[mask, ["player_id"] + Z_COLS]
        .drop_duplicates("player_id", keep="first")
        .set_index("player_id")[Z_COLS]
        .rename(columns=Z_RENAME)
    )
    return out


club_z = z_by_context(players, is_national=False)
nat_z = z_by_context(players, is_national=True)
assert club_z.shape[1] == len(FEATURES), f"Expected {len(FEATURES)} z features, got {club_z.shape[1]}"

common_ids = club_z.index.intersection(nat_z.index)
delta_z = nat_z.loc[common_ids] - club_z.loc[common_ids]
delta_z = delta_z.rename(columns={f: f"delta_{f}" for f in FEATURES})

contrib = pd.DataFrame(
    {f"contrib_{f}": delta_z[f"delta_{f}"].abs() * w[f] for f in FEATURES},
    index=delta_z.index,
)

gaps = pd.concat(
    [
        players.loc[~players["is_international"], ["player_id", "player_name"]]
        .drop_duplicates("player_id")
        .set_index("player_id"),
        delta_z,
        contrib,
    ],
    axis=1,
).reset_index()

contrib_cols = [f"contrib_{f}" for f in FEATURES]
gaps["total_weighted_gap"] = gaps[contrib_cols].sum(axis=1)

perf_gap = nat_z.loc[common_ids].mul(w, axis=1).sum(axis=1) - club_z.loc[common_ids].mul(w, axis=1).sum(axis=1)
gaps["performance_gap"] = gaps["player_id"].map(perf_gap)

print(f"Players with gap table: {len(gaps):,}")
display(gaps[["player_name", "performance_gap", "total_weighted_gap"]].head(8))

---
## 4. Global feature importance (all dual-context players)

Rank features by mean PCA-weighted absolute gap \(I_f = \text{mean}(w_f \cdot |\Delta z_f|)\).

In [ ]:
if "gaps" not in globals():
    raise RuntimeError("Run §3 first to build `gaps`.")

importance_rows = []
for f in FEATURES:
    importance_rows.append({
        "feature": f,
        "label": FEATURE_LABELS[f],
        "pca_weight": w[f],
        "mean_abs_delta_z": gaps[f"delta_{f}"].abs().mean(),
        "mean_delta_z": gaps[f"delta_{f}"].mean(),
        "importance": gaps[f"contrib_{f}"].mean(),
    })

importance = (
    pd.DataFrame(importance_rows)
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
importance["rank"] = importance.index + 1
importance["pct_of_total"] = (
    100 * importance["importance"] / importance["importance"].sum()
).round(2)

display(importance[[
    "rank", "label", "pca_weight", "mean_abs_delta_z", "mean_delta_z", "importance", "pct_of_total"
]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = importance.sort_values("importance", ascending=True)
axes[0].barh(plot_df["label"], plot_df["importance"], color="#4C72B0")
axes[0].set_xlabel("Mean weighted |Δz| (importance)")
axes[0].set_title("Feature importance — club vs national gap")
colors = ["#DD8452" if x > 0 else "#4C72B0" for x in plot_df["mean_delta_z"]]
axes[1].barh(plot_df["label"], plot_df["mean_delta_z"], color=colors)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("Mean Δz (national − club)")
axes[1].set_title("Direction: orange = higher for country")
plt.tight_layout()
plt.show()

In [ ]:
# Table and plots are in the §4 cell above.

### Interpretation — global club vs national gap

**Left chart (importance):** Which metrics contribute most to club–national differences (PCA-weighted mean \|Δz\|), regardless of direction.

**Top 3 drivers of the gap:**
1. **xG / shot** — highest importance (~0.095). Shot quality relative to peers differs most between contexts.
2. **Pass completion %** — second (~0.093). Passing efficiency vs context-specific baselines shifts widely across players.
3. **Interceptions p90** — third (~0.078). Defensive reading volume also separates club vs national profiles.

Mid-tier: clearances, xG p90, pressures, attacking-third passes/carries. **Dribbles p90** contributes least to the overall gap (smallest bars) even though it shows a large directional swing (see right chart).

**Right chart (direction):** Mean Δz = national − club. **Orange** = higher for country; **blue** = higher for club.

**Elevated for national team (orange):**
- **Clearances p90** — largest positive shift (~+0.02). International football in this sample is associated with more last-ditch defending relative to club peers.
- **xG p90** and **pressures p90** — small positive shifts. Slight national-team bumps in chance creation and pressing intensity.

**Elevated for club (blue):**
- **Dribbles p90** — largest negative shift (~−0.09). Players dribble more relative to **club** baselines than **national** baselines.
- **xG / shot**, **pass completion %**, **att. third carries/passes p90**, **shots p90** — moderate club-side shifts. Domestic league play is tied to better shot quality, passing completion, and attacking-third involvement in z-score terms.

**Takeaway:** The gap is driven mainly by **efficiency metrics** (xG/shot, pass completion) and **defensive actions** (interceptions, clearances), not raw shot volume. Directionally, players look more like **chance-quality + possession passers at club** and slightly more **defensive / clearance-heavy at national team**—consistent with tighter international games and different squad roles.

---
## 5. Specialist quadrants (Club-strong vs National-strong)

In [ ]:
if USE_PRECOMPUTED_SCORES:
    try:
        scores = spark.sql(f"SELECT * FROM {CONSISTENCY_TABLE}").toPandas()
        print(f"Loaded {len(scores):,} rows from {CONSISTENCY_TABLE}")
    except Exception as exc:
        print(f"Could not load {CONSISTENCY_TABLE}: {exc}")
        USE_PRECOMPUTED_SCORES = False

if not USE_PRECOMPUTED_SCORES:
    club_perf = club_z.mul(w, axis=1).sum(axis=1).rename("club_performance_score")
    nat_perf = nat_z.mul(w, axis=1).sum(axis=1).rename("national_performance_score")
    scores = pd.concat([club_perf, nat_perf], axis=1).reset_index()
    scores = scores.merge(
        players.loc[~players["is_international"], ["player_id", "player_name"]].drop_duplicates("player_id"),
        on="player_id",
    )
    club_med = scores["club_performance_score"].median()
    nat_med = scores["national_performance_score"].median()

    def assign_quadrant(row):
        hc = row.club_performance_score >= club_med
        hn = row.national_performance_score >= nat_med
        if hc and hn:
            return "Elite both"
        if hc and not hn:
            return "Club-strong"
        if not hc and hn:
            return "National-strong"
        return "Lower both"

    scores["performance_quadrant"] = scores.apply(assign_quadrant, axis=1)

gaps = gaps.merge(
    scores[["player_id", "performance_quadrant", "club_performance_score", "national_performance_score"]],
    on="player_id",
    how="left",
)

display(scores["performance_quadrant"].value_counts().to_frame("count"))


In [ ]:
CLUB_SPECIALIST = "Club-strong"
NAT_SPECIALIST = "National-strong"

club_spec = gaps.loc[gaps["performance_quadrant"] == CLUB_SPECIALIST]
nat_spec = gaps.loc[gaps["performance_quadrant"] == NAT_SPECIALIST]

print(f"Club-strong: {len(club_spec):,} | National-strong: {len(nat_spec):,}")

specialist_compare = []
for f in FEATURES:
    specialist_compare.append({
        "feature": f,
        "label": FEATURE_LABELS[f],
        "club_strong_mean_dz": club_spec[f"delta_{f}"].mean(),
        "nat_strong_mean_dz": nat_spec[f"delta_{f}"].mean(),
        "difference": nat_spec[f"delta_{f}"].mean() - club_spec[f"delta_{f}"].mean(),
        "club_strong_importance": club_spec[f"contrib_{f}"].mean(),
        "nat_strong_importance": nat_spec[f"contrib_{f}"].mean(),
    })

specialist_df = (
    pd.DataFrame(specialist_compare)
    .assign(abs_difference=lambda d: d["difference"].abs())
    .sort_values("abs_difference", ascending=False)
    .reset_index(drop=True)
)
specialist_df["rank_separation"] = specialist_df.index + 1

display(specialist_df[[
    "rank_separation", "label",
    "club_strong_mean_dz", "nat_strong_mean_dz", "difference",
    "club_strong_importance", "nat_strong_importance",
]])

### Interpretation — Club-strong vs National-strong

The chart compares **mean Δz (national − club)** within each specialist cohort (not individual players). Bars on opposite sides of zero show the two groups are defined by **opposite context profiles**.

**Top separating features** (largest blue–orange gaps):

1. **Att. third passes p90** — strongest discriminator. Club-strong cohort averages **negative** Δz (~−1.1); National-strong **positive** (~+0.75). Club specialists create more progressive passes relative to club baselines; international specialists do so relative to national baselines.
2. **Interceptions p90** — second. Club-strong lean club-side; National-strong lean national-side. International specialists sit in systems that reward more ball-winning in z-score terms.
3. **Att. third carries p90** — third. Same pattern: club specialists drive into the final third more at **club**; national specialists more for **country**.

Also separate: **xG p90**, **pressures p90**, **shots p90**, **dribbles p90**, **clearances p90** — same sign pattern across all eight (club-strong negative, national-strong positive).

**How this differs from §4:** §4 averages **all** dual-context players. §5 contrasts only the **extreme quadrants**, so separation is sharper (~1–2 z units between cohort means vs ~0.02–0.09 globally).

**Archetype summary:**
- **Club Specialist:** winger/full-back/progressive profiles who rank higher vs club peers (Clauss, Carrasco, Coman).
- **International Specialist:** players who rank higher vs national peers (Kanté, Gnabry, Herrera) — often more defensive/work-rate or system-dependent national roles.

In [ ]:
top_sep = specialist_df.head(8)
x = np.arange(len(top_sep))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width / 2, top_sep["club_strong_mean_dz"], width, label=CLUB_SPECIALIST, color="#4C72B0")
ax.bar(x + width / 2, top_sep["nat_strong_mean_dz"], width, label=NAT_SPECIALIST, color="#DD8452")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(top_sep["label"], rotation=35, ha="right")
ax.set_ylabel("Mean Δz (national − club)")
ax.set_title("Top separating features: Club-strong vs National-strong")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Findings — Club Specialists vs International Specialists

### 6.1 Who are the specialists?

| Quadrant | Definition |
|----------|------------|
| **Club-strong** (Club Specialist) | PCA-weighted performance ≥ median for **club**, below median for **national** |
| **National-strong** (International Specialist) | Below median for **club**, ≥ median for **national** |

These are tactical archetypes from **relative** z-scores within each context, not raw Statbomb totals.

### 6.2 Which features explain the gap overall?

See **§4** (confirmed in our run):

| Rank | Feature | Role |
|------|---------|------|
| 1 | xG / shot | Largest gap driver; **higher at club** |
| 2 | Pass completion % | Large gap; **higher at club** |
| 3 | Interceptions p90 | Large gap; roughly **club-leaning** |

National-team lifts: **clearances p90** (largest positive Δz), then xG p90 and pressures p90. **Dribbles p90** has the biggest club–national directional gap but lowest overall importance.

### 6.3 Which features separate Club-strong from National-strong?

See **§5** chart and `specialist_df` (confirmed in our run):

| Rank | Feature | Club-strong cohort | National-strong cohort |
|------|---------|--------------------|-------------------------|
| 1 | Att. third passes p90 | Mean Δz ≈ **−1.1** (club-favoured) | Mean Δz ≈ **+0.75** (national-favoured) |
| 2 | Interceptions p90 | Negative mean Δz | Positive mean Δz |
| 3 | Att. third carries p90 | Negative mean Δz | Positive mean Δz |

All top separators show **opposite signs** between cohorts: club specialists improve vs **club** z-baselines; international specialists vs **national** z-baselines on the same metrics.

### 6.4 How this links to PCA loadings

PCA weights ensure we emphasize metrics that define the **playing-style space** used in clustering, not just raw volume stats. A feature can have a moderate weight but still rank high in importance if players consistently show large \(|\Delta z|\) on it.

**Conclusion:** *Club specialists* are distinguished by **attacking-third involvement and chance creation** (passes/carries p90) relative to club peers. *International specialists* show the inverse profile—higher z-scores for **pressing, interceptions, and national-team xG/shot volume** vs national peers. The same player can look like a club winger domestically and a more defensive/system-driven international contributor, which supports RQ3: performance context matters beyond raw talent.

In [ ]:
def top_players(quadrant: str, n: int = 5) -> pd.DataFrame:
    sub = gaps.loc[gaps["performance_quadrant"] == quadrant].copy()
    if quadrant == CLUB_SPECIALIST:
        sub["specialist_score"] = sub["club_performance_score"] - sub["national_performance_score"]
    else:
        sub["specialist_score"] = sub["national_performance_score"] - sub["club_performance_score"]
    cols = ["player_name", "performance_quadrant", "club_performance_score", "national_performance_score", "specialist_score"]
    return sub.nlargest(n, "specialist_score")[cols]


print(f"Example {CLUB_SPECIALIST} players:")
display(top_players(CLUB_SPECIALIST))
print(f"\nExample {NAT_SPECIALIST} players:")
display(top_players(NAT_SPECIALIST))